In [1]:
# ===== Cell 1: 安裝環境（Colab）=====
!pip install -U pip >/dev/null
!pip install "datasets==2.21.0" "transformers==4.44.2" "accelerate==0.34.2" \
             "torchmetrics==1.4.2" "scikit-learn==1.5.2" "pandas==2.2.2" >/dev/null

import sys, platform, torch, transformers, datasets, pandas as pd
print("✅ Python:", platform.python_version())
print("✅ PyTorch:", torch.__version__)
print("✅ Transformers:", transformers.__version__)
print("✅ Datasets:", datasets.__version__)
print("✅ Pandas:", pd.__version__)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
✅ Python: 3.12.12
✅ PyTorch: 2.8.0+cu126
✅ Transformers: 4.44.2
✅ Datasets: 2.21.0
✅ Pandas: 2.2.2


In [2]:
# ===== Cell 2: 共用設定與紀錄器 =====
import os, json, time, math, random
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from torchmetrics import PearsonCorrCoef, Accuracy

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 路徑與 run_id
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ROOT = f"/content/outputs/{RUN_ID}"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(f"{ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{ROOT}/preds", exist_ok=True)

# 輔助：logger（會同時 print 並寫入 run.log）
LOG_PATH = f"{ROOT}/run.log"
def log(msg: str):
    stamp = datetime.now().strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# 輔助：CSV 紀錄器（metrics）
METRICS_CSV = f"{ROOT}/metrics.csv"
if not os.path.exists(METRICS_CSV):
    pd.DataFrame(columns=[
        "run_id","model_name","epoch","split",
        "loss_total","loss_reg","loss_cls","pearson","accuracy"
    ]).to_csv(METRICS_CSV, index=False)

def log_metrics(model_name, epoch, split, loss_total=None, loss_reg=None, loss_cls=None, pearson=None, accuracy=None):
    row = {
        "run_id": RUN_ID, "model_name": model_name, "epoch": epoch, "split": split,
        "loss_total": loss_total, "loss_reg": loss_reg, "loss_cls": loss_cls,
        "pearson": pearson, "accuracy": accuracy
    }
    df = pd.DataFrame([row])
    df.to_csv(METRICS_CSV, mode="a", header=False, index=False)

log(f"Outputs folder: {ROOT}")


[01:22:57] Outputs folder: /content/outputs/20251031_012257


In [7]:
# ===== Cell 3.1: Hotfix for dataset column names (entailment_judgment vs judgement) =====
from datasets import DatasetDict

# 檢視實際欄位
train_cols = list(raw["train"].features.keys())
val_cols   = list(raw["validation"].features.keys())
test_cols  = list(raw["test"].features.keys())
log(f"Train columns: {train_cols}")
log(f"Val   columns: {val_cols}")
log(f"Test  columns: {test_cols}")

# 候選鍵（分類 / 回歸）
CLS_CANDIDATES = ["entailment_judgment", "entailment_judgement", "entailment_label", "gold_label", "label"]
REG_CANDIDATES = ["relatedness_score", "relatedness", "score"]

def _pick_key(columns, candidates, name_for_log):
    for k in candidates:
        if k in columns:
            log(f"Detected {name_for_log} column: '{k}'")
            return k
    raise KeyError(f"Cannot find {name_for_log} in columns: {columns}")

CLS_KEY = _pick_key(train_cols, CLS_CANDIDATES, "classification label")
REG_KEY = _pick_key(train_cols, REG_CANDIDATES, "regression label")

# 重新定義 build_dataloaders 使用偵測到的鍵
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

def build_dataloaders(tokenizer_name: str, batch_size=16, max_len=256):
    tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

    def _encode(batch):
        enc = tok(
            batch["premise"], batch["hypothesis"],
            padding="max_length", truncation=True, max_length=max_len
        )
        # 用偵測到的實際欄位名取值
        enc["reg_label"] = batch[REG_KEY]
        enc["cls_label"] = batch[CLS_KEY]
        return enc

    ds_train = raw["train"].map(_encode, batched=True)
    ds_val   = raw["validation"].map(_encode, batched=True)
    ds_test  = raw["test"].map(_encode, batched=True)

    cols = ["input_ids", "attention_mask"]
    # BERT 會有 token_type_ids；RoBERTa 沒有 → 動態相容
    if "token_type_ids" in ds_train.features:
        cols.append("token_type_ids")

    ds_train.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_val.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_test.set_format(type="torch", columns=cols+["reg_label","cls_label"])

    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return tok, dl_train, dl_val, dl_test

log("Hotfix applied: build_dataloaders() now uses auto-detected label keys.")


[01:24:51] Train columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[01:24:51] Val   columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[01:24:51] Test  columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[01:24:51] Detected classification label column: 'entailment_judgment'
[01:24:51] Detected regression label column: 'relatedness_score'
[01:24:51] Hotfix applied: build_dataloaders() now uses auto-detected label keys.


In [4]:
# ===== Cell 4: 多輸出模型定義（共用） =====
from transformers import AutoModel

class MultiOutputEncoder(nn.Module):
    def __init__(self, encoder_name: str, num_labels_cls: int = 3, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        # heads
        self.reg_head = nn.Linear(hidden, 1)            # 回歸：relatedness_score
        self.cls_head = nn.Linear(hidden, num_labels_cls) # 分類：3 類

    def forward(self, batch):
        # 兼容 token_type_ids（RoBERTa 沒有）
        enc_kwargs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        }
        if "token_type_ids" in batch:
            enc_kwargs["token_type_ids"] = batch["token_type_ids"]

        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]  # [CLS] 位元（對 RoBERTa 也是第一個 token）
        x = self.dropout(pooled)
        reg = self.reg_head(x).squeeze(-1)    # (B,)
        cls = self.cls_head(x)                # (B, 3)
        return reg, cls


In [5]:
# ===== Cell 5: 訓練/驗證/測試 公用流程 =====
from tqdm.auto import tqdm

def run_train_eval(model_name_key: str,
                   epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256,
                   reg_loss_weight=1.0, cls_loss_weight=1.0, device=None):
    """
    model_name_key: 'bert' 或 'roberta'
    """
    assert model_name_key in ("bert","roberta")
    encoder_name = MODEL_NAMES[model_name_key]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Data
    tok, dl_train, dl_val, dl_test = build_dataloaders(encoder_name, batch_size=batch_size, max_len=max_len)

    # Model & Optim
    model = MultiOutputEncoder(encoder_name).to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_reg_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    best_val_score = -1e9
    best_path = f"{ROOT}/checkpoints/{model_name_key}_best.pt"

    log(f"[{model_name_key}] Start training: epochs={epochs}, lr={lr}, bs={batch_size}, max_len={max_len}")
    for ep in range(1, epochs+1):
        model.train()
        tr_loss_total = tr_loss_reg = tr_loss_cls = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_name_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)

            reg_pred, cls_logits = model(batch)
            loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
            loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
            loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            tr_loss_total += loss.item()
            tr_loss_reg   += loss_r.item()
            tr_loss_cls   += loss_c.item()

        n_batches = len(dl_train)
        log_metrics(model_name_key, ep, "train",
                    loss_total=tr_loss_total/n_batches,
                    loss_reg=tr_loss_reg/n_batches,
                    loss_cls=tr_loss_cls/n_batches)

        # ---- Validation ----
        model.eval()
        va_loss_total = va_loss_reg = va_loss_cls = 0.0
        pearson_metric = PearsonCorrCoef().to(device)
        acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

        all_reg_pred, all_reg_true = [], []
        all_cls_pred, all_cls_true = [], []

        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_name_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                reg_pred, cls_logits = model(batch)

                loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
                loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
                loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

                va_loss_total += loss.item()
                va_loss_reg   += loss_r.item()
                va_loss_cls   += loss_c.item()

                # metrics
                pearson_metric.update(reg_pred, batch["reg_label"].float())
                acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

                all_reg_pred.extend(reg_pred.detach().cpu().tolist())
                all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
                all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
                all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

        n_val = len(dl_val)
        pear = float(pearson_metric.compute().detach().cpu())
        acc  = float(acc_metric.compute().detach().cpu())
        log_metrics(model_name_key, ep, "val",
                    loss_total=va_loss_total/n_val,
                    loss_reg=va_loss_reg/n_val,
                    loss_cls=va_loss_cls/n_val,
                    pearson=pear, accuracy=acc)
        log(f"[{model_name_key}] ep{ep:02d} | Val Pearson={pear:.4f}, Acc={acc:.4f}")

        # 保存最佳
        score = pear + acc   # 你也可以改成加權：0.5*pear + 0.5*acc
        if score > best_val_score:
            best_val_score = score
            torch.save(model.state_dict(), best_path)
            # 同時保留當前 val 預測
            pd.DataFrame({
                "y_reg_true": all_reg_true,
                "y_reg_pred": all_reg_pred,
                "y_cls_true": all_cls_true,
                "y_cls_pred": all_cls_pred
            }).to_csv(f"{ROOT}/preds/val_{model_name_key}.csv", index=False)
            log(f"[{model_name_key}] ✅ Best updated. Saved to: {best_path}")

    # ---- Test with best checkpoint ----
    log(f"[{model_name_key}] Loading best ckpt: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    pearson_metric = PearsonCorrCoef().to(device)
    acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

    all_reg_pred, all_reg_true = [], []
    all_cls_pred, all_cls_true = [], []

    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_name_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            pearson_metric.update(reg_pred, batch["reg_label"].float())
            acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

            all_reg_pred.extend(reg_pred.detach().cpu().tolist())
            all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
            all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
            all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

    pear = float(pearson_metric.compute().detach().cpu())
    acc  = float(acc_metric.compute().detach().cpu())
    log_metrics(model_name_key, epoch=0, split="test", pearson=pear, accuracy=acc)
    pd.DataFrame({
        "y_reg_true": all_reg_true,
        "y_reg_pred": all_reg_pred,
        "y_cls_true": all_cls_true,
        "y_cls_pred": all_cls_pred
    }).to_csv(f"{ROOT}/preds/test_{model_name_key}.csv", index=False)
    log(f"[{model_name_key}] ✅ Test | Pearson={pear:.4f}, Acc={acc:.4f}")

    return {
        "best_val_score": best_val_score,
        "test_pearson": pear, "test_acc": acc,
        "best_ckpt": best_path
    }


In [16]:
# ===== Cell 6: 跑 BERT-base =====
res_bert = run_train_eval(
    model_name_key="bert",
    epochs=6,           # 正式跑可以調高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[bert] Done. {json.dumps(res_bert, indent=2)}")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[03:16:23] [bert] Start training: epochs=6, lr=2e-05, bs=16, max_len=256


Train[bert] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[bert] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

[03:19:28] [bert] ep01 | Val Pearson=0.8615, Acc=0.8500
[03:19:30] [bert] ✅ Best updated. Saved to: /content/outputs/20251031_012257/checkpoints/bert_best.pt


...

Valid[bert] ep6:   0%|          | 0/32 [00:00<?, ?it/s]

[03:35:01] [bert] ep06 | Val Pearson=0.8661, Acc=0.8540
[03:35:01] [bert] Loading best ckpt: /content/outputs/20251031_012257/checkpoints/bert_best.pt


Test[bert]:   0%|          | 0/308 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f994f550680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f994f550680>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
Traceback (most recent call last):
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[03:36:03] [bert] ✅ Test | Pearson=0.8708, Acc=0.8683
[03:36:03] [bert] Done. {
  "best_val_score": 1.7475404739379883,
  "test_pearson": 0.870833694934845,
  "test_acc": 0.868276834487915,
  "best_ckpt": "/content/outputs/20251031_012257/checkpoints/bert_best.pt"
}


In [17]:
# ===== Cell 7: 跑 RoBERTa-base =====
res_roberta = run_train_eval(
    model_name_key="roberta",
    epochs=6,           # 正式跑可以調高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[roberta] Done. {json.dumps(res_roberta, indent=2)}")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[03:36:27] [roberta] Start training: epochs=6, lr=2e-05, bs=16, max_len=256


Train[roberta] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[roberta] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

...

Train[roberta] ep6:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[roberta] ep6:   0%|          | 0/32 [00:00<?, ?it/s]

[03:56:02] [roberta] ep06 | Val Pearson=0.8982, Acc=0.8980
[03:56:02] [roberta] Loading best ckpt: /content/outputs/20251031_012257/checkpoints/roberta_best.pt


Test[roberta]:   0%|          | 0/308 [00:00<?, ?it/s]

[03:57:05] [roberta] ✅ Test | Pearson=0.8990, Acc=0.8884
[03:57:05] [roberta] Done. {
  "best_val_score": 1.8002323508262634,
  "test_pearson": 0.8989531397819519,
  "test_acc": 0.8883702158927917,
  "best_ckpt": "/content/outputs/20251031_012257/checkpoints/roberta_best.pt"
}


In [10]:
# ===== Cell 8: 匯總表（方便貼進報告）=====
df = pd.read_csv(METRICS_CSV)
# 取 test 分數
test_rows = df[(df["split"]=="test") & (df["epoch"]==0)][["model_name","pearson","accuracy"]]
summary = test_rows.copy().reset_index(drop=True)
summary.to_csv(f"{ROOT}/summary.csv", index=False)
log("Saved summary:\n" + summary.to_string(index=False))
summary


[01:50:20] Saved summary:
model_name  pearson  accuracy
      bert 0.878469  0.868480
   roberta 0.892706  0.880455


,model_name,pearson,accuracy
0,bert,0.878469,0.868480
1,roberta,0.892706,0.880455


# GPT-2

In [11]:
# ===== Cell G2-1: Build GPT-2 dataloaders =====
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

def build_dataloaders_gpt2(tokenizer_name: str = "gpt2", batch_size=16, max_len=256):
    tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
    # GPT-2 沒有 pad，統一用 eos 當 pad
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def _encode(batch):
        # 句對以 eos 當分隔
        texts = [p + tok.eos_token + h for p, h in zip(batch["premise"], batch["hypothesis"])]
        enc = tok(texts, padding="max_length", truncation=True, max_length=max_len, add_special_tokens=True)
        enc["reg_label"] = batch[REG_KEY]
        enc["cls_label"] = batch[CLS_KEY]
        return enc

    ds_train = raw["train"].map(_encode, batched=True)
    ds_val   = raw["validation"].map(_encode, batched=True)
    ds_test  = raw["test"].map(_encode, batched=True)

    cols = ["input_ids", "attention_mask"]  # GPT-2 不使用 token_type_ids
    ds_train.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_val.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_test.set_format(type="torch", columns=cols+["reg_label","cls_label"])

    # 為了避免 colab 的 multiprocessing 噪音，這裡 num_workers=0（穩定）
    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return tok, dl_train, dl_val, dl_test

log("GPT-2 dataloader builder ready.")


[02:31:57] GPT-2 dataloader builder ready.


In [12]:
# ===== Cell G2-2: GPT-2 multi-output model =====
import torch
from torch import nn
from transformers import GPT2Model, GPT2Config

class MultiOutputGPT2(nn.Module):
    def __init__(self, encoder_name: str = "gpt2", num_labels_cls: int = 3, dropout: float = 0.1):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(encoder_name)
        # pad_token_id 對齊 tokenizer（上個 cell 已把 pad 設為 eos）
        if self.gpt2.config.pad_token_id is None:
            self.gpt2.config.pad_token_id = self.gpt2.config.eos_token_id
        hidden = self.gpt2.config.n_embd
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(hidden, 1)
        self.cls_head = nn.Linear(hidden, num_labels_cls)

    def forward(self, batch):
        out = self.gpt2(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        last_h = out.last_hidden_state  # (B, L, H)
        # masked mean pooling
        mask = batch["attention_mask"].unsqueeze(-1).float()  # (B, L, 1)
        summed = (last_h * mask).sum(dim=1)                   # (B, H)
        denom = mask.sum(dim=1).clamp(min=1e-6)               # (B, 1)
        pooled = summed / denom
        x = self.dropout(pooled)
        reg = self.reg_head(x).squeeze(-1)   # (B,)
        cls = self.cls_head(x)               # (B,3)
        return reg, cls

log("MultiOutputGPT2 ready.")


[02:32:05] MultiOutputGPT2 ready.


In [13]:
# ===== Cell G2-3: Train & evaluate GPT-2 =====
from torch.optim import AdamW
from torchmetrics import PearsonCorrCoef, Accuracy
from tqdm.auto import tqdm
import pandas as pd
import json

def run_train_eval_gpt2(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256,
                        reg_loss_weight=1.0, cls_loss_weight=1.0, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders_gpt2(batch_size=batch_size, max_len=max_len)

    model = MultiOutputGPT2("gpt2").to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_reg_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    best_val_score = -1e9
    best_path = f"{ROOT}/checkpoints/gpt2_best.pt"
    model_key = "gpt2"

    log(f"[{model_key}] Start training: epochs={epochs}, lr={lr}, bs={batch_size}, max_len={max_len}")
    for ep in range(1, epochs+1):
        # ---- train ----
        model.train()
        tr_loss_total = tr_loss_reg = tr_loss_cls = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
            loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
            loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_total += loss.item()
            tr_loss_reg   += loss_r.item()
            tr_loss_cls   += loss_c.item()

        n_batches = len(dl_train)
        log_metrics(model_key, ep, "train",
                    loss_total=tr_loss_total/n_batches,
                    loss_reg=tr_loss_reg/n_batches,
                    loss_cls=tr_loss_cls/n_batches)

        # ---- valid ----
        model.eval()
        va_loss_total = va_loss_reg = va_loss_cls = 0.0
        pearson_metric = PearsonCorrCoef().to(device)
        acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

        all_reg_pred, all_reg_true = [], []
        all_cls_pred, all_cls_true = [], []

        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                reg_pred, cls_logits = model(batch)
                loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
                loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
                loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

                va_loss_total += loss.item()
                va_loss_reg   += loss_r.item()
                va_loss_cls   += loss_c.item()

                pearson_metric.update(reg_pred, batch["reg_label"].float())
                acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

                all_reg_pred.extend(reg_pred.detach().cpu().tolist())
                all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
                all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
                all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

        pear = float(pearson_metric.compute().detach().cpu())
        acc  = float(acc_metric.compute().detach().cpu())
        log_metrics(model_key, ep, "val",
                    loss_total=va_loss_total/len(dl_val),
                    loss_reg=va_loss_reg/len(dl_val),
                    loss_cls=va_loss_cls/len(dl_val),
                    pearson=pear, accuracy=acc)
        log(f"[{model_key}] ep{ep:02d} | Val Pearson={pear:.4f}, Acc={acc:.4f}")

        score = pear + acc
        if score > best_val_score:
            best_val_score = score
            torch.save(model.state_dict(), best_path)
            pd.DataFrame({
                "y_reg_true": all_reg_true,
                "y_reg_pred": all_reg_pred,
                "y_cls_true": all_cls_true,
                "y_cls_pred": all_cls_pred
            }).to_csv(f"{ROOT}/preds/val_{model_key}.csv", index=False)
            log(f"[{model_key}] ✅ Best updated. Saved to: {best_path}")

    # ---- test ----
    log(f"[{model_key}] Loading best ckpt: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    pearson_metric = PearsonCorrCoef().to(device)
    acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

    all_reg_pred, all_reg_true = [], []
    all_cls_pred, all_cls_true = [], []

    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            pearson_metric.update(reg_pred, batch["reg_label"].float())
            acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

            all_reg_pred.extend(reg_pred.detach().cpu().tolist())
            all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
            all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
            all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

    pear = float(pearson_metric.compute().detach().cpu())
    acc  = float(acc_metric.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", pearson=pear, accuracy=acc)
    pd.DataFrame({
        "y_reg_true": all_reg_true, "y_reg_pred": all_reg_pred,
        "y_cls_true": all_cls_true, "y_cls_pred": all_cls_pred
    }).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test | Pearson={pear:.4f}, Acc={acc:.4f}")

    return {"best_val_score": best_val_score, "test_pearson": pear, "test_acc": acc, "best_ckpt": best_path}


In [15]:
# ===== Cell G2-4: Run GPT-2 =====
res_gpt2 = run_train_eval_gpt2(
    epochs=8,       # 正式評分可拉高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[gpt2] Done. {json.dumps(res_gpt2, indent=2)}")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[02:45:08] [gpt2] Start training: epochs=8, lr=2e-05, bs=16, max_len=256


Train[gpt2] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[gpt2] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

[02:48:42] [gpt2] ep01 | Val Pearson=0.7330, Acc=0.7980
[02:48:54] [gpt2] ✅ Best updated. Saved to: /content/outputs/20251031_012257/checkpoints/gpt2_best.pt


...

Train[gpt2] ep8:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[gpt2] ep8:   0%|          | 0/32 [00:00<?, ?it/s]

[03:14:36] [gpt2] ep08 | Val Pearson=0.8431, Acc=0.8440
[03:14:36] [gpt2] Loading best ckpt: /content/outputs/20251031_012257/checkpoints/gpt2_best.pt


Test[gpt2]:   0%|          | 0/308 [00:00<?, ?it/s]

[03:15:48] [gpt2] ✅ Test | Pearson=0.8531, Acc=0.8608
[03:15:48] [gpt2] Done. {
  "best_val_score": 1.7079685926437378,
  "test_pearson": 0.8531033992767334,
  "test_acc": 0.8607671856880188,
  "best_ckpt": "/content/outputs/20251031_012257/checkpoints/gpt2_best.pt"
}


In [ ]:
# ===== Cell G2-5: Append GPT-2 to summary =====
import pandas as pd
df = pd.read_csv(METRICS_CSV)
test_rows = df[(df["split"]=="test") & (df["epoch"]==0)][["model_name","pearson","accuracy"]]
summary = test_rows.reset_index(drop=True)
summary.to_csv(f"{ROOT}/summary.csv", index=False)
log("Updated summary:\n" + summary.to_string(index=False))
summary


# Multi-task

In [18]:
# ===== Cell M1: Single-task BERT models (regression / classification) =====
import torch
from torch import nn
from transformers import AutoModel

class BertRegressor(nn.Module):
    def __init__(self, encoder_name="bert-base-uncased", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, batch):
        enc_kwargs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]}
        if "token_type_ids" in batch: enc_kwargs["token_type_ids"] = batch["token_type_ids"]
        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        reg = self.head(x).squeeze(-1)
        return reg

class BertClassifier(nn.Module):
    def __init__(self, encoder_name="bert-base-uncased", num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, num_labels)

    def forward(self, batch):
        enc_kwargs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]}
        if "token_type_ids" in batch: enc_kwargs["token_type_ids"] = batch["token_type_ids"]
        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        logits = self.head(x)
        return logits


In [19]:
# ===== Cell M2: Train/Eval single-task runners (share logs/metrics) =====
from torch.optim import AdamW
from torchmetrics import PearsonCorrCoef, Accuracy
from tqdm.auto import tqdm
import pandas as pd, json, torch

def run_bert_regression(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders("bert-base-uncased", batch_size=batch_size, max_len=max_len)
    model = BertRegressor().to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    model_key = "bert_reg"
    best_val = 1e9
    best_path = f"{ROOT}/checkpoints/{model_key}_best.pt"

    log(f"[{model_key}] start training")
    for ep in range(1, epochs+1):
        model.train(); tr = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            pred = model(batch)
            loss = loss_fn(pred, batch["reg_label"].float())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += loss.item()
        log_metrics(model_key, ep, "train", loss_total=tr/len(dl_train), loss_reg=tr/len(dl_train))

        # valid
        model.eval(); va = 0.0; pear_m = PearsonCorrCoef().to(device)
        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                pred = model(batch)
                loss = loss_fn(pred, batch["reg_label"].float())
                va += loss.item()
                pear_m.update(pred, batch["reg_label"].float())
        pear = float(pear_m.compute().detach().cpu())
        log_metrics(model_key, ep, "val", loss_total=va/len(dl_val), loss_reg=va/len(dl_val), pearson=pear)
        if va < best_val:
            best_val = va
            torch.save(model.state_dict(), best_path)
            log(f"[{model_key}] ✅ best updated -> {best_path}")

    # test
    log(f"[{model_key}] load best: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device)); model.eval()
    pear_m = PearsonCorrCoef().to(device)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            pred = model(batch)
            pear_m.update(pred, batch["reg_label"].float())
            y_true += batch["reg_label"].detach().cpu().tolist()
            y_pred += pred.detach().cpu().tolist()
    pear = float(pear_m.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", pearson=pear)
    pd.DataFrame({"y_reg_true": y_true, "y_reg_pred": y_pred}).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test Pearson={pear:.4f}")
    return {"test_pearson": pear, "best_ckpt": best_path}

def run_bert_classification(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders("bert-base-uncased", batch_size=batch_size, max_len=max_len)
    model = BertClassifier().to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    model_key = "bert_cls"
    best_val = -1.0
    best_path = f"{ROOT}/checkpoints/{model_key}_best.pt"

    log(f"[{model_key}] start training")
    for ep in range(1, epochs+1):
        model.train(); tr = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            logits = model(batch)
            loss = loss_fn(logits, batch["cls_label"].long())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += loss.item()
        log_metrics(model_key, ep, "train", loss_total=tr/len(dl_train), loss_cls=tr/len(dl_train))

        # valid
        model.eval(); va = 0.0; acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                logits = model(batch)
                loss = loss_fn(logits, batch["cls_label"].long())
                va += loss.item()
                acc_m.update(logits.softmax(-1), batch["cls_label"])
        acc = float(acc_m.compute().detach().cpu())
        log_metrics(model_key, ep, "val", loss_total=va/len(dl_val), loss_cls=va/len(dl_val), accuracy=acc)
        if acc > best_val:
            best_val = acc
            torch.save(model.state_dict(), best_path)
            log(f"[{model_key}] ✅ best updated -> {best_path}")

    # test
    log(f"[{model_key}] load best: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device)); model.eval()
    acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            logits = model(batch)
            acc_m.update(logits.softmax(-1), batch["cls_label"])
            y_true += batch["cls_label"].detach().cpu().tolist()
            y_pred += logits.argmax(-1).detach().cpu().tolist()
    acc = float(acc_m.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", accuracy=acc)
    pd.DataFrame({"y_cls_true": y_true, "y_cls_pred": y_pred}).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test Acc={acc:.4f}")
    return {"test_acc": acc, "best_ckpt": best_path}


In [20]:
# ===== Cell M3: Run single-task baselines =====
res_reg = run_bert_regression(epochs=3, lr=2e-5, batch_size=16, max_len=256)
res_cls = run_bert_classification(epochs=3, lr=2e-5, batch_size=16, max_len=256)
log(f"[bert_reg] {json.dumps(res_reg, indent=2)}")
log(f"[bert_cls] {json.dumps(res_cls, indent=2)}")


Map:   0%|          | 0/4927 [00:00<?, ? examples/s]

[04:27:48] [bert_reg] start training


Train[bert_reg] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[bert_reg] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

[04:31:03] [bert_reg] ✅ best updated -> /content/outputs/20251031_012257/checkpoints/bert_reg_best.pt


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f994f550680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f994f550680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Valid[bert_cls] ep3:   0%|          | 0/32 [00:00<?, ?it/s]

[04:48:11] [bert_cls] load best: /content/outputs/20251031_012257/checkpoints/bert_cls_best.pt


Test[bert_cls]:   0%|          | 0/308 [00:00<?, ?it/s]

[04:49:14] [bert_cls] ✅ Test Acc=0.8579
[04:49:14] [bert_reg] {
  "test_pearson": 0.8859186172485352,
  "best_ckpt": "/content/outputs/20251031_012257/checkpoints/bert_reg_best.pt"
}
[04:49:14] [bert_cls] {
  "test_acc": 0.8579257130622864,
  "best_ckpt": "/content/outputs/20251031_012257/checkpoints/bert_cls_best.pt"
}


In [21]:
# ===== Cell M4: Build comparison summary =====
import pandas as pd

df = pd.read_csv(METRICS_CSV)

want = df[(df["split"]=="test") & (df["epoch"]==0)][
    ["model_name","pearson","accuracy"]
].fillna("")

# 只挑出 bert (multi-output), bert_reg, bert_cls
mask = want["model_name"].isin(["bert","bert_reg","bert_cls"])
summary_bert = want[mask].rename(columns={
    "model_name":"model",
    "pearson":"test_pearson",
    "accuracy":"test_acc"
}).reset_index(drop=True)

summary_bert.to_csv(f"{ROOT}/summary_bert_multi_vs_single.csv", index=False)
log("Saved summary_bert_multi_vs_single.csv")
summary_bert


[04:49:20] Saved summary_bert_multi_vs_single.csv


,model,test_pearson,test_acc
0,bert,0.878469,0.86848
1,bert,0.870834,0.868277
2,bert_reg,0.885919,
3,bert_cls,,0.857926


In [1]:
# ===== Cell EA-1: load preds and build analysis table =====
import os, pandas as pd
from datasets import load_dataset

MODEL_FOR_ANALYSIS = "roberta"   # 可改成 "bert" / "gpt2"
TEST_PREDS = f"{ROOT}/preds/test_{MODEL_FOR_ANALYSIS}.csv"
assert os.path.exists(TEST_PREDS), f"找不到 {TEST_PREDS}，請先完成該模型的測試推論。"

# 讀測試預測
preds = pd.read_csv(TEST_PREDS)

# 讀原始資料（保持與前面一致）
raw = load_dataset("SemEvalWorkshop/sem_eval_2014_task_1")

# 把 datasets 的欄位拉出來做 DataFrame
df_raw = pd.DataFrame({
    "premise": raw["test"]["premise"],
    "hypothesis": raw["test"]["hypothesis"],
    "y_reg_true": raw["test"][REG_KEY],
    "y_cls_true": raw["test"][CLS_KEY]
})

# 合併
df = pd.concat([df_raw.reset_index(drop=True), preds.reset_index(drop=True)], axis=1)

# 派生欄位
df["reg_residual"] = df["y_reg_pred"] - df["y_reg_true"]
df["reg_abs_err"]  = df["reg_residual"].abs()
df["cls_correct"]  = (df["y_cls_pred"] == df["y_cls_true"]).astype(int)

# 儲存完整分析表
OUT_CSV = f"{ROOT}/error_analysis_{MODEL_FOR_ANALYSIS}.csv"
df.to_csv(OUT_CSV, index=False)
log(f"[EA] Saved analysis table: {OUT_CSV}")
df.head(3)


NameError: name 'ROOT' is not defined